In [1]:
import requests
import geopandas as gpd
from shapely.geometry import shape
import sys

if 'workhorse' not in sys.executable.split('/'):
    origin = 'workspace/'
    sys.path.append('/media/')
else:
    origin = 'data/Aldhani/eoagritwin/'
    sys.path.append('/home/potzschf/repos/')
from helperToolz.helpsters import path_safe

url = "https://owsproxy.lgl-bw.de/owsproxy/wfs/WFS_LW-BW_GISELA_landw_Parzellen"

storPath = path_safe('/data/Aldhani/eoagritwin/fields/01_IACS/1_Polygons/BW/')

year = 2018

In [2]:
params = {
    "request": "GetFeature",
    "service": "WFS",
    "version": "2.0.0",
    "outputFormat": "json",
    "typeNames": f"lw:v_gisela_landw_parzellen_{year}",
    "count": 2000,
    "startIndex": 0
}

In [3]:
r = requests.get(url, params=params, timeout=120)
data = r.json()

In [8]:
data.get("crs", {}).get("properties", {}).get("name", None)

'urn:ogc:def:crs:EPSG::25832'

In [9]:
output_file = f"{storPath}BW_Parzellen_{year}.gpkg"
layer_name = f"Parzellen{year}"
first_batch = True
batch = 0
total = 0

while True:
    print(f"Lade Batch {batch+1}, startIndex={params['startIndex']} ...")

    r = requests.get(url, params=params, timeout=120)
    data = r.json()

    features = data.get("features", [])
    if not features:
        print("Keine weiteren Features – fertig!")
        break

    geometries = [shape(f["geometry"]) for f in features]
    properties = [f["properties"] for f in features]

    gdf = gpd.GeoDataFrame(properties, geometry=geometries, crs="EPSG:25832")

    if first_batch:
        gdf.to_file(output_file, layer=layer_name, driver="GPKG", engine="pyogrio")
        first_batch = False
    else:
        gdf.to_file(output_file, layer=layer_name, driver="GPKG", engine="pyogrio", mode="a")

    total += len(features)
    print(f"  → {len(features)} Features gespeichert, gesamt: {total}")

    params["startIndex"] += len(features)
    batch += 1

print(f"\nFertig! {total} Features gespeichert in '{output_file}'")

Lade Batch 1, startIndex=0 ...
  → 2000 Features gespeichert, gesamt: 2000
Lade Batch 2, startIndex=2000 ...
  → 2000 Features gespeichert, gesamt: 4000
Lade Batch 3, startIndex=4000 ...
  → 2000 Features gespeichert, gesamt: 6000
Lade Batch 4, startIndex=6000 ...
  → 2000 Features gespeichert, gesamt: 8000
Lade Batch 5, startIndex=8000 ...
  → 2000 Features gespeichert, gesamt: 10000
Lade Batch 6, startIndex=10000 ...
  → 2000 Features gespeichert, gesamt: 12000
Lade Batch 7, startIndex=12000 ...
  → 2000 Features gespeichert, gesamt: 14000
Lade Batch 8, startIndex=14000 ...
  → 2000 Features gespeichert, gesamt: 16000
Lade Batch 9, startIndex=16000 ...
  → 2000 Features gespeichert, gesamt: 18000
Lade Batch 10, startIndex=18000 ...
  → 2000 Features gespeichert, gesamt: 20000
Lade Batch 11, startIndex=20000 ...
  → 2000 Features gespeichert, gesamt: 22000
Lade Batch 12, startIndex=22000 ...
  → 2000 Features gespeichert, gesamt: 24000
Lade Batch 13, startIndex=24000 ...
  → 2000 Feat